# StoryZop: Instagram Story Visual Analysis

This notebook runs the complete StoryZop pipeline on Google Colab.

**Before running:**
1. Change runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Add your Instagram `sessionid` to **Colab Secrets** (🔑 icon on the left sidebar):
   - Name: `INSTAGRAM_SESSIONID`
   - Value: *(your sessionid cookie value)*
   - Toggle: Enable notebook access

## 1. Install Dependencies & Clone Repo

In [ ]:
# Core dependencies
!pip install -q playwright Pillow pydantic pydantic-settings python-dotenv nest-asyncio

# Install Chromium WITH system dependencies (critical for Colab)
!playwright install --with-deps chromium

# AI / Vision dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers>=4.57.0 accelerate qwen-vl-utils bitsandbytes easyocr

# Clone the repo
!git clone https://github.com/10Unknownboy/StoryZop.git 2>/dev/null || echo 'Repo already cloned'
%cd StoryZop

# Enable async in Colab
import nest_asyncio
nest_asyncio.apply()

print('\n\u2705 All dependencies installed.')

## 2. GPU Check

In [ ]:
from src.vision.gpu import GPUManager

gpu_info = GPUManager.detect_gpu()
GPUManager.print_gpu_status()

print(f"\n4B (4-bit) fits: {GPUManager.estimate_model_fit('4b', quantized=True)}")
print(f"8B (4-bit) fits: {GPUManager.estimate_model_fit('8b', quantized=True)}")
print(f"32B (4-bit) fits: {GPUManager.estimate_model_fit('32b', quantized=True)}")

vram = gpu_info['vram_total_gb']
if vram < 15:
    print(f"\n\u26a0\ufe0f T4 detected ({vram:.1f}GB). Models will be loaded one at a time with 4-bit quantization.")
elif vram < 24:
    print(f"\n\u2705 L4/A10 detected ({vram:.1f}GB). 4B + 8B will fit.")
else:
    print(f"\n\u2705 A100/H100 detected ({vram:.1f}GB). All models should fit.")

## 3. Configuration

In [ ]:
from src.config import get_config

session_id = None
try:
    from google.colab import userdata
    session_id = userdata.get('INSTAGRAM_SESSIONID')
    print('\u2705 Session ID loaded from Colab Secrets.')
except Exception:
    print('\u26a0\ufe0f Could not read INSTAGRAM_SESSIONID from Colab Secrets.')

config = get_config(
    instagram_sessionid=session_id,
    use_4bit_quantization=True,
    headless=True,
)

print(f'Data dir: {config.data_dir}')
print(f'Models: {config.initial_model} | {config.primary_model}')

## 4. Initialize Database

In [ ]:
from src.database.database import Database

db = Database(config.db_path)
db.initialize()

state = db.get_processing_state()
print(f'\u2705 Database ready at {config.db_path}')
print(f'   Completed: {len(state["completed"])} | Pending: {len(state["pending"])} | Incomplete: {len(state["incomplete"])}')

## 5. Load AI Models

In [ ]:
from src.vision.qwen4b import Qwen4BScreener
from src.vision.qwen8b import Qwen8BAnalyzer
from src.vision.qwen32b import Qwen32BExpert
from src.vision.ocr import OCREngine

screener = Qwen4BScreener(config)
analyzer = Qwen8BAnalyzer(config)

expert = None
if GPUManager.estimate_model_fit('32b', quantized=config.use_4bit_quantization):
    expert = Qwen32BExpert(config)
    print('\u2705 32B Expert enabled (will load on demand).')
else:
    print('\u26a0\ufe0f 32B Expert skipped (not enough VRAM).')

ocr_engine = OCREngine(
    languages=config.ocr_languages,
    confidence_threshold=config.ocr_confidence_threshold
)
print(f'\u2705 OCR engine ready (available: {ocr_engine.is_available})')

# Quick verify: load 4B, confirm it works, then free VRAM
print('\nVerifying 4B model access...')
screener.load_model()
print('\u2705 4B loaded successfully!')
screener.unload_model()
print('\u2705 4B unloaded. Models will reload during pipeline run.\n')

print('--- All models ready ---')

## 6. Launch Browser & Authenticate

In [ ]:
from src.browser.session import BrowserSession
from src.browser.instagram import InstagramNavigator
from src.browser.stories import StoryNavigator
from src.capture.frame_manager import FrameManager
from src.capture.sampler import StorySampler

session = BrowserSession(config)
await session.launch()
print('\u2705 Browser launched.')

if config.instagram_sessionid:
    await session.load_sessionid(config.instagram_sessionid)
    print('\u2705 Session ID cookie injected.')
else:
    print('\u26a0\ufe0f No session ID provided.')

instagram_nav = InstagramNavigator(session.page, config)
story_nav = StoryNavigator(session.page, config)
frame_manager = FrameManager(config)
sampler = StorySampler(config, frame_manager)

await instagram_nav.navigate_to_instagram()
is_auth = await instagram_nav.verify_authentication()
print(f'\nAuthenticated: {is_auth}')

if is_auth:
    await instagram_nav.dismiss_dialogs()
    print('\u2705 Ready to scan stories.')
else:
    print('\u274c Authentication failed. Check your session ID.')

## 6b. Debug: Screenshot of what the browser sees

This shows you what Instagram looks like to the headless browser. Useful for debugging story discovery.

In [ ]:
from IPython.display import display, Image as IPImage
import os

os.makedirs(config.data_dir, exist_ok=True)
debug_path = str(config.data_dir / 'debug_feed.png')

await session.page.screenshot(path=debug_path, full_page=False)
print(f'Screenshot saved to {debug_path}')
display(IPImage(filename=debug_path, width=412))

## 6c. Debug: Test story discovery separately

Run this cell to test story discovery before running the full pipeline.

In [ ]:
# Test story discovery
discovered = await instagram_nav.get_stories_tray()
print(f'Found {len(discovered)} stories:')
for s in discovered:
    print(f"  @{s['username']} (index={s['index']})")

if not discovered:
    print('\n\u26a0\ufe0f No stories found. Possible reasons:')
    print('  1. Your session ID has expired — get a fresh one from your browser')
    print('  2. None of the accounts you follow have active stories right now')
    print('  3. Instagram is showing a different page layout')
    print('\nCheck the screenshot above to see what the browser sees.')

## 7. Run the Full Pipeline

In [ ]:
from src.pipeline import StoryPipeline

pipeline = StoryPipeline(config, db)
pipeline.set_browser(session)
pipeline.set_navigators(instagram_nav, story_nav)
pipeline.set_sampler(sampler, frame_manager)
pipeline.set_models(screener, analyzer, expert)
pipeline.set_ocr(ocr_engine)

print('Starting pipeline...\n')
stats = await pipeline.run()

print(f'\n--- Pipeline Complete ---')
print(f'  Discovered: {stats["discovered"]}')
print(f'  Completed:  {stats["completed"]}')
print(f'  Revisited:  {stats["revisited"]}')
print(f'  Failed:     {stats["failed"]}')

## 8. View Results

In [ ]:
from src.analysis.report import ReportGenerator

report_gen = ReportGenerator(db)

report = report_gen.generate_text_report()
if report:
    print(report)
else:
    print('No stories analyzed yet.')

## 9. Export Data

In [ ]:
import os
os.makedirs(config.data_dir, exist_ok=True)

json_path = config.data_dir / 'export.json'
csv_path = config.data_dir / 'export.csv'

report_gen.export_json(json_path)
report_gen.export_csv(csv_path)

print(f'\u2705 JSON exported to {json_path}')
print(f'\u2705 CSV exported to {csv_path}')

try:
    from google.colab import files
    files.download(str(json_path))
    files.download(str(csv_path))
except ImportError:
    pass

## 10. Cleanup

In [ ]:
await session.close()
print('\u2705 Browser closed.')

if screener.is_loaded:
    screener.unload_model()
if analyzer.is_loaded:
    analyzer.unload_model()
if expert and expert.is_loaded:
    expert.unload_model()
print('\u2705 Models unloaded.')

db.close()
print('\u2705 Database closed.')